# Dataset Reproduction

This notebook rebuilds the `K3_regions` anonymised workbook from the original myVal v2 workbook and, when the release that was supplied for the project is available, checks it against that release.

> **Hypothesis test: anonymisation cost (classification).** The hypothesis notebooks read `data/raw/Household_Insurance_Synthetic_Dataset_v2_K3_regions.xlsx`. This notebook shows how that release follows from `data/raw/myVal_Synthetic_Datasets_release_v2.xlsx`, so that the input of the experiment is documented and reproducible. It does not change any hypothesis notebook.

The rebuild has two stages. Stage A turns the original workbook into the de-branded and pseudonymised workbook (23 sheets). Stage B applies the regions generalisation with k = 3 enforced on the full combination of state, age, gender and occupation. The result is written to `data/raw`, where the hypothesis notebooks read it, and is compared cell by cell with the supplied release when that file is present.

## Why This Release Is Used

The experiments of the hypothesis test compare a source release with an anonymised release, so the anonymised release has to be one that can be defended as anonymised and that can be traced back to the original data. The release follows the usual approach of combining pseudonymisation, generalisation and local suppression to reach a k-anonymity criterion (Samarati, 2001, Sweeney, 2002). Direct identifiers are replaced with keyed-hash pseudonyms, the quasi-identifiers (state, age band, gender and occupation of the customer, and the location of the property) are coarsened, and where a combination of quasi-identifiers still describes fewer than 3 customers the least informative attribute is suppressed first, in the order occupation, age, gender and state. This is the procedure described in the methodology draft, where the criterion of k = 3 is also stated to be weak, since it does not protect against attribute disclosure or against linkage on attributes outside the listed sets. The question of the hypothesis test is the effect of the transformation on predictive utility and not the strength of the privacy guarantee.

Three versions of the anonymised workbook were prepared and compared in the comparison report of the team. All three start from the same de-branded and pseudonymised workbook and differ in the last generalisation step.

| Records in groups smaller than 3 | Pseudonymised only | K3 | K3_regions |
|---|---|---|---|
| Customers: state, age, gender, occupation (full combination) | 481 of 500 | 280 of 500 (128 unique) | 0 |
| Customers: state, age, gender | 50 | 0 | 0 |
| Customers: age, gender, occupation | 270 | 0 | 0 |
| Customers: state, occupation | 189 | 0 | 0 |
| Properties: state, type, bedrooms | 111 | 0 | 0 |
| Properties: latitude, longitude | 488 | 0 | 0 |

- **Pseudonymised only.** Replacing identifiers does not change how unique the customers are on their demographic attributes, so 481 of 500 customers remain in groups smaller than 3.
- **K3.** This version enforced groups of at least 3 on one attribute set at a time. An attacker who knows all four attributes of a customer faces the full combination, and there 280 customers were still in groups of one or two. The state of a customer whose state had been blurred was still visible through the property record, the time zone and 168 notification messages, and the suppression was heavy and uneven, so that blurred values were themselves informative.
- **K3_regions.** This version enforces k = 3 directly on the full combination, which also covers every subset of it, merges the states into three regions, and removes the leaks through the property record and the notification texts. Its cost is a coarse geography and fewer age and occupation categories, so that questions about single states or jobs cannot be answered from it. The team report found no measurable loss of model performance for this version, and it is therefore the release used in the hypothesis test.

The K3 version is kept only as an internal variant and is not used in any experiment. The rules below are those that reproduce the K3_regions release exactly. The earlier K3 release is discussed again in the limitations at the end of this notebook.

## Data Sources

Two files are required. The original workbook is read from `data/raw`, and the pseudonym mapping `k3_regions_pseudonym_mapping.parquet` is read from `references` (see the section on the pseudonym mapping). The rebuilt workbook is written to `data/raw` under the name that the hypothesis notebooks read. Writing is refused when a file with the same name already exists there, so that an existing release cannot be overwritten by accident.

The supplied release is optional. When the file is placed in `data/` (one level above `raw`), it is read once as the reference of the comparison at the end and is never modified. Without it the workbook is still rebuilt, and only the comparison and the comparison of the privacy check are skipped.

In [1]:
# -------------------------------------------------------------------
# Data Sources
# -------------------------------------------------------------------
import collections
import re
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd

RELEASE_NAME = "Household_Insurance_Synthetic_Dataset_v2_K3_regions.xlsx"

# Original workbook, input of the rebuild.
ORIGINAL_PATH = Path("../../../data/raw/myVal_Synthetic_Datasets_release_v2.xlsx")

# Pseudonym mapping (original identifier -> pseudonym of the supplied release), input of Stage A.
MAPPING_PATH = Path("../../../references/k3_regions_pseudonym_mapping.parquet")

# Supplied release (optional, only used for the comparison at the end, never written).
REFERENCE_PATH = Path("../../../data") / RELEASE_NAME

# Rebuilt release, written by this notebook.
OUTPUT_PATH = Path("../../../data/raw") / RELEASE_NAME

# Set to True only to replace an existing rebuilt file.
OVERWRITE = False

# Set to True to make the pseudonym mapping file again from the supplied release (needs the supplied release in data/).
REBUILD_MAPPING = False

HAS_REFERENCE = REFERENCE_PATH.exists()

required = [("Original workbook", ORIGINAL_PATH)]
if not REBUILD_MAPPING:
    required.append(("Pseudonym mapping", MAPPING_PATH))
elif not HAS_REFERENCE:
    raise FileNotFoundError(f"REBUILD_MAPPING needs the supplied release: {REFERENCE_PATH}")

for label, path in required:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    print(f"{label:<20} found: {path}")

if HAS_REFERENCE:
    print(f"{'Supplied release':<20} found: {REFERENCE_PATH}")
else:
    print(f"{'Supplied release':<20} not found, the comparison with it is skipped: {REFERENCE_PATH}")

if OUTPUT_PATH.exists() and not OVERWRITE:
    raise FileExistsError(f"{OUTPUT_PATH} already exists. Set OVERWRITE = True to replace it.")

print(f"{'Output':<20} will be written to: {OUTPUT_PATH}")

Original workbook    found: ..\..\..\data\raw\myVal_Synthetic_Datasets_release_v2.xlsx
Pseudonym mapping    found: ..\..\..\references\k3_regions_pseudonym_mapping.parquet
Supplied release     not found, the comparison with it is skipped: ..\..\..\data\Household_Insurance_Synthetic_Dataset_v2_K3_regions.xlsx
Output               will be written to: ..\..\..\data\raw\Household_Insurance_Synthetic_Dataset_v2_K3_regions.xlsx


## Workbook Structure

The workbooks are read once into memory as lists of rows, with the values exactly as stored in the file. The original workbook has 26 sheets and the release has 23 sheets (when the supplied release is present), because the sheets `Contents`, `Schema Verification` and `Sheet1` are not part of the release.

In [2]:
# -------------------------------------------------------------------
# Workbook Loading
# -------------------------------------------------------------------

def read_book(path):
    workbook = openpyxl.load_workbook(path, read_only=True)
    return {sheet.title: [tuple(row) for row in sheet.iter_rows(values_only=True)] for sheet in workbook.worksheets}


original = read_book(ORIGINAL_PATH)
reference = read_book(REFERENCE_PATH) if HAS_REFERENCE else None

print(f"Original workbook: {len(original)} sheets")

if reference is None:
    print("Supplied release:  not available, the comparison with it is skipped")
    print("\nSheets of the original workbook (rows including the header row):")
    for number, sheet in enumerate(original, start=1):
        print(f"{number:02d}. {sheet:<20} original {len(original[sheet]):>7,}")
else:
    print(f"Supplied release:  {len(reference)} sheets")

    print("\nSheets only in the original workbook:")
    for sheet in original:
        if sheet not in reference:
            print(f"- {sheet}")

    print("\nSheet inventory (rows including the header row):")
    for number, sheet in enumerate(reference, start=1):
        original_rows = len(original[sheet]) if sheet in original else 0
        print(f"{number:02d}. {sheet:<20} original {original_rows:>7,} | release {len(reference[sheet]):>7,}")


Original workbook: 26 sheets
Supplied release:  not available, the comparison with it is skipped

Sheets of the original workbook (rows including the header row):
01. Overview             original      45
02. Contents             original      22
03. Schema Verification  original      16
04. Data Dictionary      original     333
05. customers            original     501
06. customer_profiles    original     501
07. properties           original     796
08. base_categories      original       8
09. assets               original   8,310
10. asset_images         original   8,310
11. asset_attachments    original   2,050
12. ai_analysis          original  24,635
13. ai_raw_responses     original   8,225
14. valuations           original   8,095
15. property_videos      original     220
16. user_activity        original  77,648
17. policies             original     423
18. pds_documents        original      25
19. pds_sections         original     439
20. claims               original      

## Stage A: De-Branded And Pseudonymised Workbook

The first stage applies the rules that are common to every anonymised release. References to the client and the platform are removed, dates are reduced to the day, property coordinates are rounded to two decimals, the AI provider and model names are replaced by neutral labels, a few columns are renamed, three sheets are dropped, and the Data Dictionary loses four decision columns and its client-specific wording.

Two inputs of this stage are not a function of the original workbook. The pseudonyms of `Customer_ID`, `Policy_Number` and `Serial_Number` were produced with a salted hash whose salt is not available, so they are taken from the mapping file (see the next section). The text of the `Overview` sheet is new prose and is written out below.

In [3]:
# -------------------------------------------------------------------
# Stage A Settings
# -------------------------------------------------------------------

DROP_SHEETS = {"Contents", "Schema Verification", "Sheet1"}

OVERVIEW_ROWS = [
    (None,),
    ("Purpose",),
    (
        "Fully synthetic relational dataset modelling a household contents-insurance app: customers, properties, "
        "documented assets, images and AI analysis, valuations, policies, product disclosure documents, claims, "
        "notifications, feedback and assistant interactions. No real customer records are included.",
    ),
    (None,),
    ("Notes",),
    (
        "Direct identifiers have been pseudonymised or removed; location and timestamps are coarsened. "
        "See the Data Dictionary tab for column definitions.",
    ),
]

MODEL_NAMES = {
    "gpt-4o-2024-05-13": "model-a-v1",
    "gpt-4o-2024-08-06": "model-a-v2",
    "gpt-4.1-2025-04-14": "model-b-v1",
}

ID_COLUMNS = ("Customer_ID", "Policy_Number", "Serial_Number")
URL_COLUMNS = {"Image_URL", "Document_URL"}
BRANDED_TEXT = {("notifications", "Message_Text"), ("ai_interactions", "Assistant_Response")}

## Pseudonym Mapping

The pseudonyms are read from the file `k3_regions_pseudonym_mapping.parquet`, a table with the columns `Column`, `Original_Value` and `Release_Value` (500 customer, 422 policy and 87 serial-number identifiers). The table was made once from the supplied release by aligning the rows of the `customers`, `policies` and `assets` sheets with the same rows of the original workbook (row order is preserved in every sheet), which is what the function `build_id_maps` below does. The cell checks that every identifier of the original workbook has a pseudonym and that the mapping is one-to-one, and the mapping is then applied to every other sheet that carries these identifiers. When the supplied release is present, the cell also checks that the file equals the mapping read from the release. Setting `REBUILD_MAPPING = True` in the data sources cell makes the file again from the supplied release (which must be in `data/`), for example when the file is lost.

In [4]:
# -------------------------------------------------------------------
# Pseudonym Mapping
# -------------------------------------------------------------------

ID_SHEETS = (("customers", "Customer_ID"), ("policies", "Policy_Number"), ("assets", "Serial_Number"))


def load_id_maps(path, source):
    table = pd.read_parquet(path)
    maps = {column: dict(zip(group["Original_Value"], group["Release_Value"])) for column, group in table.groupby("Column")}
    for sheet, column in ID_SHEETS:
        index = source[sheet][0].index(column)
        identifiers = {row[index] for row in source[sheet][1:] if row[index] is not None}
        missing = identifiers - set(maps[column])
        if missing:
            raise ValueError(f"{len(missing)} identifiers of {column} have no pseudonym in the mapping file")
        if len(set(maps[column].values())) != len(maps[column]):
            raise ValueError(f"The pseudonyms of {column} are not one-to-one")
    return maps


def build_id_maps(source, release):
    """How the mapping file was made: row i of the original sheet is row i of the supplied release."""
    maps = {}
    for sheet, column in ID_SHEETS:
        source_index = source[sheet][0].index(column)
        release_index = release[sheet][0].index(column)
        mapping = {}
        for source_row, release_row in zip(source[sheet][1:], release[sheet][1:]):
            if source_row[source_index] is not None:
                mapping[source_row[source_index]] = release_row[release_index]
        maps[column] = mapping
    return maps


if REBUILD_MAPPING:
    rows = [(column, key, value) for column, mapping in build_id_maps(original, reference).items() for key, value in mapping.items()]
    MAPPING_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows, columns=["Column", "Original_Value", "Release_Value"]).to_parquet(MAPPING_PATH, index=False)
    print(f"Mapping file written: {MAPPING_PATH}\n")

id_maps = load_id_maps(MAPPING_PATH, original)

print("PSEUDONYM MAPPING")
print("-" * 80)
for column, mapping in id_maps.items():
    print(f"{column:<15} {len(mapping):>6,} identifiers mapped one-to-one")

if reference is not None:
    print(f"\nMapping file equals the mapping read from the supplied release: {build_id_maps(original, reference) == id_maps}")


PSEUDONYM MAPPING
--------------------------------------------------------------------------------
Customer_ID        500 identifiers mapped one-to-one
Policy_Number      422 identifiers mapped one-to-one
Serial_Number       87 identifiers mapped one-to-one


In [5]:
# -------------------------------------------------------------------
# Stage A Functions
# -------------------------------------------------------------------

def neutral_text(value):
    """De-branding of the Data Dictionary wording."""
    if not isinstance(value, str):
        return value
    value = value.replace("cdn.myval.example", "cdn.example.test").replace("myval_app", "platform_app")
    value = value.replace("openai_vision", "vendor_a_vision").replace("joining myVal", "joining the platform")
    value = value.replace("Had_myVal_Inventory", "Had_Inventory_Record")
    return re.sub(r"\b[Pp]roduction\b", "source system", value)


def whole_number(value):
    return int(value) if float(value).is_integer() else value


def stage_a(source, maps):
    result = {}
    for sheet, rows in source.items():
        if sheet in DROP_SHEETS:
            continue

        if sheet == "Overview":
            result[sheet] = [list(row) for row in OVERVIEW_ROWS]
            continue

        header = list(rows[0])

        if sheet == "Data Dictionary":
            keep = [header.index(c) for c in ("Dataset", "Column Name", "Type", "Allowed Values / Format", "Description")]
            table = [[neutral_text(row[i]) for i in keep] for row in rows]
            table[0] = [rows[0][i] for i in keep]
            result[sheet] = table
            continue

        position = {name: i for i, name in enumerate(header)}
        table = [[("Had_Inventory_Record" if name == "Had_myVal_Inventory" else name) for name in header]]

        for row in rows[1:]:
            row = list(row)

            for column in ID_COLUMNS:
                if column in position and row[position[column]] is not None:
                    row[position[column]] = maps[column][row[position[column]]]

            if sheet == "customers":
                for column in ("Account_Created", "Last_Active_At"):
                    if isinstance(row[position[column]], str):
                        row[position[column]] = row[position[column]][:10]

            for column in ("Postcode", "Address_Postcode"):
                if column in position and isinstance(row[position[column]], str):
                    row[position[column]] = int(row[position[column]])

            if sheet == "properties":
                for column in ("Latitude", "Longitude"):
                    if row[position[column]] is not None:
                        row[position[column]] = whole_number(round(float(row[position[column]]), 2))

            if sheet == "assets" and isinstance(row[position["Name_Edited_By_Customer"]], bool):
                row[position["Name_Edited_By_Customer"]] = int(row[position["Name_Edited_By_Customer"]])

            for column in URL_COLUMNS & set(position):
                if isinstance(row[position[column]], str):
                    row[position[column]] = row[position[column]].replace("cdn.myval.example", "cdn.example.test")

            if "Provider" in position and row[position["Provider"]] == "openai":
                row[position["Provider"]] = "vendor_a"

            if "Model_Version" in position and row[position["Model_Version"]] in MODEL_NAMES:
                row[position["Model_Version"]] = MODEL_NAMES[row[position["Model_Version"]]]

            if sheet == "ai_raw_responses" and isinstance(row[position["Raw_Response_JSON"]], str):
                text = row[position["Raw_Response_JSON"]].replace("chatcmpl-", "resp-")
                for old, new in MODEL_NAMES.items():
                    text = text.replace(old, new)
                row[position["Raw_Response_JSON"]] = re.sub(
                    r'"system_fingerprint": "[^"]*"', '"system_fingerprint": "fp_redacted"', text
                )

            if sheet in ("claims", "claim_events"):
                for column in ("Lodgement_Channel", "Channel"):
                    if column in position and row[position[column]] == "myval_app":
                        row[position[column]] = "platform_app"

            for branded_sheet, column in BRANDED_TEXT:
                if sheet == branded_sheet and isinstance(row[position[column]], str):
                    row[position[column]] = row[position[column]].replace("myVal ", "")

            table.append(row)

        result[sheet] = table
    return result


processed = stage_a(original, id_maps)

print("STAGE A COMPLETE")
print("-" * 80)
print(f"Sheets after stage A: {len(processed)}")
print(f"Rows in customers:    {len(processed['customers']) - 1:,}")
print(f"Rows in properties:   {len(processed['properties']) - 1:,}")

STAGE A COMPLETE
--------------------------------------------------------------------------------
Sheets after stage A: 23
Rows in customers:    500
Rows in properties:   795


## Stage B: Regions Generalisation

The second stage generalises the quasi-identifiers and enforces k = 3 on their full combination. It is computed from the original values and not from any other release, because values that are blurred in a release cannot be recovered from it.

- **Customers.** State is merged into three regions (East Coast, North, West & South), age into three bands (18-34, 35-54, 55+) and occupation into four groups. Postcode is replaced by `xxxx` and the city by the region. Customers in a group of fewer than 3 on the combination of state, age, gender and occupation are blurred one level at a time, in the order occupation, age, gender and state, until no group has fewer than 3 customers. The value "Prefer not to say" is never rewritten. If the fully blurred group is still smaller than 3, one customer is added to it, namely the first customer in file order of the largest group.
- **Properties.** The region replaces the state, rare property types and bedroom counts are merged by the same level-by-level rule on region, type and bedrooms, the coordinates are replaced by the centre of the region, the time zone becomes `Australia`, the valuation is rounded to the nearest 5,000 and occupants and bedrooms are top-coded at 6 and 5.
- **Other tables.** Languages spoken by fewer than 10 customers become `Other`, state abbreviations in notification texts become the region of the customer (`your area` when the state of the customer is blurred), and the label of the object in the raw AI responses is shortened.

In [6]:
# -------------------------------------------------------------------
# Stage B Settings
# -------------------------------------------------------------------

K = 3
NOT_STATED = "Not stated"
RARE_LANGUAGE_MIN = 10

REGION = {
    "ACT": "East Coast", "NSW": "East Coast", "TAS": "East Coast", "VIC": "East Coast",
    "NT": "North", "QLD": "North",
    "SA": "West & South", "WA": "West & South",
}

AGE_BAND = {
    "18-24": "18-34", "25-34": "18-34",
    "35-44": "35-54", "45-54": "35-54",
    "55-64": "55+", "65+": "55+",
}

OCCUPATION_GROUPS = {
    "Business & Technology": [
        "Accountant", "Bank Officer", "Bookkeeper", "Civil Engineer", "Data Analyst", "Graphic Designer",
        "Marketing Coordinator", "Project Manager", "Real Estate Agent", "Retail Manager",
        "Self-employed Consultant", "Software Engineer",
    ],
    "Health & Education": [
        "Aged Care Worker", "Childcare Worker", "Paramedic", "Physiotherapist",
        "Primary School Teacher", "Registered Nurse", "University Lecturer",
    ],
    "Services & Other": ["Barista", "Chef", "Hairdresser", "Farmer", "Police Officer", "Retired", "Student"],
    "Trades & Transport": ["Carpenter", "Electrician", "Mechanic", "Plumber", "Truck Driver", "Warehouse Supervisor"],
}
OCCUPATION = {job: group for group, jobs in OCCUPATION_GROUPS.items() for job in jobs}

PROPERTY_TYPE = {
    "apartment": "apartment", "duplex": "attached", "semi-detached": "attached",
    "villa": "attached", "house": "house", "terrace": "terrace",
}

STATE_PATTERN = re.compile(r"\b(ACT|NSW|NT|QLD|SA|TAS|VIC|WA)\b")

In [7]:
# -------------------------------------------------------------------
# Stage B Functions
# -------------------------------------------------------------------

def group_sizes(frame):
    return frame.groupby(list(frame.columns), dropna=False)[frame.columns[0]].transform("size").to_numpy()


def blur_customers(base):
    """Blur level of every customer: 1 occupation, 2 plus age, 3 plus gender, 4 plus state."""
    level = np.zeros(len(base), dtype=int)

    def effective():
        frame = base.copy()
        frame.loc[level >= 1, "O"] = "*"
        frame.loc[level >= 2, "A"] = "*"
        frame.loc[level >= 3, "G"] = NOT_STATED
        frame.loc[level >= 4, "S"] = "*"
        return frame

    while True:
        small = group_sizes(effective()) < K
        if not small.any():
            return level
        if (level[small] < 4).any():
            level[small & (level < 4)] += 1
            continue
        # The fully blurred group is still too small: add the first customer of the largest base group.
        sizes = base.groupby(["S", "A", "G", "O"])["S"].transform("size").to_numpy()
        candidates = np.where(level < 4)[0]
        level[candidates[np.argsort(-sizes[candidates], kind="stable")][0]] = 4


def blur_properties(frame, inherit_state):
    """Type, bedrooms and state are blurred in this order for properties in a group smaller than K."""
    count = len(frame)
    type_blurred = np.zeros(count, dtype=bool)
    bedrooms_blurred = np.zeros(count, dtype=bool)
    state_blurred = inherit_state.copy()

    while True:
        effective = pd.DataFrame({
            "S": np.where(state_blurred, "*", frame["S"]),
            "T": np.where(type_blurred, "*", frame["T"]),
            "B": np.where(bedrooms_blurred, -1, frame["B"]),
        })
        small = group_sizes(effective) < K
        if not small.any():
            return type_blurred, bedrooms_blurred, state_blurred
        moved = False
        for i in np.where(small)[0]:
            if not type_blurred[i]:
                type_blurred[i] = True
                moved = True
            elif not bedrooms_blurred[i]:
                bedrooms_blurred[i] = True
                moved = True
            elif not state_blurred[i]:
                state_blurred[i] = True
                moved = True
        if not moved:
            return type_blurred, bedrooms_blurred, state_blurred


def round_half_up(values, base):
    return np.floor(values / base + 0.5) * base


def regions_release(frames):
    """Generalised customers, customer_profiles, properties and notifications, computed from the original values."""
    result = {}

    customers = frames["customers"].copy()
    state = customers["State"].map(REGION)
    age = customers["Age_Band"].map(AGE_BAND)
    occupation = customers["Occupation"].map(OCCUPATION)
    gender = customers["Gender"]
    if (state.isna() | age.isna() | occupation.isna()).any():
        raise ValueError("A state, age band or occupation has no group")

    base = pd.DataFrame({"S": state.values, "A": age.values, "G": gender.values, "O": occupation.values})
    level = blur_customers(base)
    new_state = np.where(level >= 4, NOT_STATED, state)
    customers["State"] = new_state
    customers["Age_Band"] = np.where(level >= 2, "All ages", age)
    customers["Gender"] = np.where(level >= 3, NOT_STATED, gender)
    customers["Occupation"] = np.where(level >= 1, "Other / not stated", occupation)
    customers["City"] = "Region: " + pd.Series(new_state, index=customers.index)
    customers["Postcode"] = "xxxx"
    result["customers"] = customers
    result["customer_level"] = level
    customer_state = dict(zip(customers["Customer_ID"], new_state))

    profiles = frames["customer_profiles"].copy()
    counts = profiles["Language_At_Home"].value_counts()
    rare = counts[counts < RARE_LANGUAGE_MIN].index
    profiles["Language_At_Home"] = profiles["Language_At_Home"].where(~profiles["Language_At_Home"].isin(rare), "Other")
    result["customer_profiles"] = profiles
    result["rare_languages"] = sorted(rare)

    properties = frames["properties"].copy()
    region = properties["Address_State"].map(REGION)
    keys = pd.DataFrame({
        "S": region.values,
        "T": properties["Property_Type"].map(PROPERTY_TYPE).values,
        "B": properties["Number_Of_Bedrooms"].astype(int).clip(upper=5).values,
    })
    inherit = properties["Customer_ID"].map(customer_state).eq(NOT_STATED).to_numpy()
    type_blurred, bedrooms_blurred, state_blurred = blur_properties(keys, inherit)
    new_region = np.where(state_blurred, NOT_STATED, region)
    properties["Property_Type"] = np.where(type_blurred, "other", keys["T"])
    properties["Number_Of_Bedrooms"] = np.where(bedrooms_blurred, np.nan, keys["B"]).astype(float)
    properties["Number_Of_Occupants"] = properties["Number_Of_Occupants"].astype(int).clip(upper=6)
    properties["Address_State"] = new_region
    properties["Address_Locality"] = "Region: " + pd.Series(new_region, index=properties.index)
    properties["Address_Postcode"] = "xxxx"
    properties["Timezone"] = "Australia"

    latitude = properties["Latitude"].astype(float)
    longitude = properties["Longitude"].astype(float)
    centres = {
        name: (round(float(latitude[region == name].mean()), 1), round(float(longitude[region == name].mean()), 1))
        for name in sorted(set(REGION.values()))
    }
    properties["Latitude"] = [centres[x][0] if x in centres else np.nan for x in new_region]
    properties["Longitude"] = [centres[x][1] if x in centres else np.nan for x in new_region]
    properties["Property_Valuation_AUD"] = round_half_up(properties["Property_Valuation_AUD"].astype(float), 5000)
    result["properties"] = properties
    result["centres"] = centres

    notifications = frames["notifications"].copy()
    owner_region = notifications["Customer_ID"].map(customer_state)

    def regional_text(text, owner):
        if not isinstance(text, str):
            return text
        text = text.replace("myVal ", "")
        return STATE_PATTERN.sub("your area" if owner == NOT_STATED else str(owner), text)

    notifications["Message_Text"] = [regional_text(t, o) for t, o in zip(notifications["Message_Text"], owner_region)]
    result["notifications"] = notifications
    return result

In [8]:
# -------------------------------------------------------------------
# Stage B Application
# -------------------------------------------------------------------

def to_native(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value


def stage_b(processed_book, source):
    names = ["customers", "customer_profiles", "properties", "notifications"]
    frames = {name: pd.DataFrame(source[name][1:], columns=source[name][0]).astype(object) for name in names}
    release = regions_release(frames)

    book = {sheet: [list(row) for row in rows] for sheet, rows in processed_book.items()}

    def put(sheet, column, values, cast=lambda x: x):
        index = book[sheet][0].index(column)
        for number, value in enumerate(values, start=1):
            value = to_native(value)
            book[sheet][number][index] = None if value is None else cast(value)

    for column in ("Age_Band", "Gender", "City", "State", "Postcode", "Occupation"):
        put("customers", column, release["customers"][column])
    put("customer_profiles", "Language_At_Home", release["customer_profiles"]["Language_At_Home"])

    properties = release["properties"]
    for column in ("Property_Type", "Address_Locality", "Address_State", "Address_Postcode", "Timezone"):
        put("properties", column, properties[column])
    for column in ("Number_Of_Occupants", "Number_Of_Bedrooms", "Property_Valuation_AUD"):
        put("properties", column, properties[column], int)
    for column in ("Latitude", "Longitude"):
        put("properties", column, properties[column], lambda x: float(round(x, 1)))
    put("notifications", "Message_Text", release["notifications"]["Message_Text"])

    index = book["ai_raw_responses"][0].index("Raw_Response_JSON")
    for row in book["ai_raw_responses"][1:]:
        if isinstance(row[index], str):
            row[index] = row[index].replace('"object": "chat.completion"', '"object": "completion"')
    return book, release


rebuilt, release_details = stage_b(processed, original)

levels = collections.Counter(release_details["customer_level"].tolist())

print("STAGE B COMPLETE")
print("-" * 80)
print(f"Customers by blur level (0 = none, 4 = state also blurred): {dict(sorted(levels.items()))}")
print(f"Languages merged into Other:                                  {release_details['rare_languages']}")
print(f"Region centres (latitude, longitude):                         {release_details['centres']}")

STAGE B COMPLETE
--------------------------------------------------------------------------------
Customers by blur level (0 = none, 4 = state also blurred): {0: 461, 1: 22, 2: 11, 3: 3, 4: 3}
Languages merged into Other:                                  ['Cantonese', 'Greek', 'Italian']
Region centres (latitude, longitude):                         {'East Coast': (-36.3, 148.1), 'North': (-23.1, 146.7), 'West & South': (-33.4, 127.6)}


## Writing The Workbook

The rebuilt workbook is written with `openpyxl`, which stores empty cells as empty strings, and is then read back from disk, so that the comparison below tests the file that was actually written and not the in-memory tables.

In [9]:
# -------------------------------------------------------------------
# Write The Workbook
# -------------------------------------------------------------------

if OUTPUT_PATH.exists() and not OVERWRITE:
    raise FileExistsError(f"{OUTPUT_PATH} already exists. Set OVERWRITE = True to replace it.")

workbook = openpyxl.Workbook()
workbook.remove(workbook.active)

for sheet, rows in rebuilt.items():
    worksheet = workbook.create_sheet(sheet)
    for row in rows:
        worksheet.append(["" if value is None else value for value in row])

workbook.save(OUTPUT_PATH)
print(f"Workbook written: {OUTPUT_PATH}")

written = read_book(OUTPUT_PATH)
print(f"Sheets read back from disk: {len(written)}")

Workbook written: ..\..\..\data\raw\Household_Insurance_Synthetic_Dataset_v2_K3_regions.xlsx
Sheets read back from disk: 23


## Cell-By-Cell Comparison With The Supplied Release

Every cell of every sheet is compared in value and in type, so that numbers stored as text, integers stored as decimals and blank cells are all distinguished. The columns holding pseudonyms and the text of the `Overview` sheet are counted separately, because the pseudonyms are an input taken from the mapping file, which was made from the supplied release, and the text of the `Overview` sheet is written out literally. The comparison is skipped when the supplied release is not in `data/`.

In [10]:
# -------------------------------------------------------------------
# Cell Comparison
# -------------------------------------------------------------------

if reference is None:
    total_mismatch = None
    print("COMPARISON SKIPPED: the supplied release is not in data/, so the rebuilt workbook is not compared with it.")
else:
    records = []
    id_cells = id_mismatch = overview_cells = overview_mismatch = 0

    print(f"Sheet names and order identical: {list(written) == list(reference)}")

    for sheet in reference:
        rebuilt_rows, reference_rows = written[sheet], reference[sheet]
        header = reference_rows[0]
        cells = mismatches = 0
        causes = collections.Counter()

        if len(rebuilt_rows) != len(reference_rows):
            records.append(dict(sheet=sheet, rows=len(reference_rows), cells=0, mismatches=-1, note="row count differs"))
            continue

        for row_number, (a_row, b_row) in enumerate(zip(rebuilt_rows, reference_rows)):
            a_row = list(a_row) + [None] * (len(b_row) - len(a_row))
            for column_number, (a, b) in enumerate(zip(a_row, b_row)):
                cells += 1
                column = header[column_number] if column_number < len(header) else "?"
                is_id = sheet != "Overview" and column in ID_COLUMNS and row_number > 0
                id_cells += is_id
                overview_cells += sheet == "Overview"
                if a != b or type(a) is not type(b):
                    mismatches += 1
                    id_mismatch += is_id
                    overview_mismatch += sheet == "Overview"
                    causes[column] += 1

        records.append(dict(sheet=sheet, rows=len(reference_rows), cells=cells, mismatches=mismatches,
                            note=", ".join(f"{k}: {v}" for k, v in causes.items())))

    comparison = pd.DataFrame(records)
    display(comparison)

    total_cells = int(comparison["cells"].sum())
    total_mismatch = int(comparison["mismatches"].clip(lower=0).sum())

    print("\nCOMPARISON SUMMARY")
    print("-" * 80)
    print(f"Cells compared:                              {total_cells:>10,}")
    print(f"Cells that differ:                           {total_mismatch:>10,}")
    print(f"Pseudonym cells (input from the release):    {id_cells:>10,} | differ {id_mismatch}")
    print(f"Overview cells (literal text):               {overview_cells:>10,} | differ {overview_mismatch}")
    print(f"Derived cells (all others):                  {total_cells - id_cells - overview_cells:>10,} | differ {total_mismatch - id_mismatch - overview_mismatch}")

COMPARISON SKIPPED: the supplied release is not in data/, so the rebuilt workbook is not compared with it.


## Privacy Check On The Rebuilt Release

The guarantee of the release is that every customer is in a group of at least 3 on the full combination of state, age band, gender and occupation, which also covers every subset of these attributes. The check below counts the groups of the rebuilt and of the supplied customer tables.

In [11]:
# -------------------------------------------------------------------
# K-Anonymity Check
# -------------------------------------------------------------------

QUASI_IDENTIFIERS = ["State", "Age_Band", "Gender", "Occupation"]


def customer_table(book):
    rows = book["customers"]
    return pd.DataFrame(rows[1:], columns=rows[0])


print("FOUR-WAY COMBINATION (State, Age_Band, Gender, Occupation)")
print("-" * 80)

books = [("Rebuilt", written)] + ([("Supplied", reference)] if reference is not None else [])

for label, book in books:
    table = customer_table(book)
    groups = table.groupby(QUASI_IDENTIFIERS, dropna=False).size()
    sizes = table.merge(groups.rename("n").reset_index(), on=QUASI_IDENTIFIERS)["n"]
    print(
        f"{label:<9} groups {len(groups):>3} | smallest group {groups.min()} | "
        f"customers in groups below {K}: {int((sizes < K).sum())} | unique customers: {int((sizes == 1).sum())}"
    )

if total_mismatch is None:
    print("\nCOMPARISON WITH THE SUPPLIED RELEASE SKIPPED")
else:
    print("\n" + ("ALL CELLS IDENTICAL" if total_mismatch == 0 else "DIFFERENCES FOUND, see the table above"))

FOUR-WAY COMBINATION (State, Age_Band, Gender, Occupation)
--------------------------------------------------------------------------------
Rebuilt   groups  71 | smallest group 3 | customers in groups below 3: 0 | unique customers: 0

COMPARISON WITH THE SUPPLIED RELEASE SKIPPED


## Inputs Not Derived From The Original Workbook

- **Pseudonyms.** `Customer_ID`, `Policy_Number` and `Serial_Number` were pseudonymised with a salted hash, and the salt is not available. The pseudonyms are therefore read from the mapping file, which was made from the supplied release for rows aligned to the original workbook. The mapping is one-to-one and is used consistently on every sheet. Without this file, or the supplied release it was made from, the workbook cannot be rebuilt with the same pseudonyms.
- **Overview text.** The text of the `Overview` sheet is new prose and is written out in the settings of stage A.

## Limitations

- One choice of the generalisation is not forced by the data. When the fully blurred group is smaller than 3, one customer is added to it, and the rule that the first customer in file order of the largest group is chosen reproduces the supplied release, but the data do not show that no other choice would. A different choice would change the values of one customer.
- The comparison is made on the values of the cells. The rebuilt file is not identical to the supplied file at the level of bytes, because the supplied file was saved again by a spreadsheet program, which changes time stamps and internal parts of the file and not the cell values.
- The earlier K3 release, which was replaced by this one, cannot be reproduced exactly from the original workbook, because its suppression was applied to several overlapping attribute sets in an order that cannot be recovered. It is not used in any experiment.